# 06 · Reviewer questions: preliminary DFCI analyses

This notebook evaluates lead time, discrimination and platinum calibration in the held-out DFCI cohort. Run notebook 04 first with the updated score-export cell. Patient-level scores stay in the protected data tree; only aggregate tables and PDFs are written to `figures/CAIA/COMPASS/reviewer_questions`. These are preliminary analyses, not manuscript-ready conclusions. The later sections require the DFCI source files and a verified demographics source if race or ethnicity is to be reported.

The primary landmark is 180 days for every model family, including the checklist, matching Figure 6. Lead-time AUC compares cases diagnosed 0–180, 181–365, 366–730 and 731–1,825 days after the landmark with event-free controls followed through 730 days or the end of the case's band, whichever is later; an all-event-free sensitivity analysis is also reported. Selection by follow-up and competing death can bias this AUC, so it should not be called unbiased.

In [ ]:
import os, sys, json
from pathlib import Path
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'survival_common').is_dir()), None)
if ROOT is None: raise RuntimeError('Run from the repository or COMPASS/notebooks')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'COMPASS' / 'federated_models'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import reviewer_questions as rq
from survival_common.federated_inference import load_bundle, merge_bundles, predict_survival
from survival_common import federated_endpoint_transfer as fet

DATA_ROOT = Path(os.getenv('COMPASS_DATA_ROOT', '/data/gusev/USERS/jpconnor/data/CAIA/COMPASS'))
RHINO_ROOT = Path(os.getenv('COMPASS_RHINO_ROOT', str(DATA_ROOT / 'rhino_data_v2')))
TRANSFER_DIR = Path(os.getenv('COMPASS_FEDERATED_TRANSFER_NO_DFCI', str(RHINO_ROOT / 'federated_transfer_adt_no_DFCI')))
SCORES_PATH = Path(os.getenv('COMPASS_REVIEWER_SCORES', str(TRANSFER_DIR / 'all_models' / 'transfer_scores.csv')))
OUT = Path(os.getenv('COMPASS_FIG_ROOT', '/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS')) / 'reviewer_questions'
# Set COMPASS_REVIEWER_BOOTSTRAPS=100 for a quick first pass; report 1000.
N_BOOT = int(os.getenv('COMPASS_REVIEWER_BOOTSTRAPS', '1000'))
SEED = 0
PRIMARY_LANDMARK = 180  # every family, as in Figure 6
MIN_EVENTS = 10
CONFIG = os.getenv('COMPASS_REVIEWER_CONFIG', 'both')
print('Scores:', SCORES_PATH, '\nAggregate output:', OUT, '\nBootstrap draws:', N_BOOT)

## Input and model identity checks

Notebook 04 writes all model scores under `all_models/transfer_scores.csv`. Its `transfer_run.json` records the frozen bundle paths used to generate the primary model scores. Change `COMPASS_REVIEWER_CONFIG` only after inspecting the delivered configurations. One patient per endpoint, family, landmark, configuration and horizon is required; duplicate model identities stop execution rather than silently pooling models.

In [ ]:
if not SCORES_PATH.exists():
    raise FileNotFoundError(f'{SCORES_PATH} missing. Rerun the updated notebook 04 to export scores under the protected data root.')
scores = pd.read_csv(SCORES_PATH, dtype={'person_id': 'string'})
scores['person_id'] = pd.to_numeric(scores.person_id, errors='raise').astype('Int64').astype('string')
required = {'person_id', 'endpoint', 'model_family', 'landmark_days', 'config', 'risk_score', 'duration_days', 'event'}
missing = required - set(scores)
if missing: raise ValueError(f'Missing score columns: {sorted(missing)}')
scores = scores.loc[scores.config.eq(CONFIG)].copy()
if scores.empty: raise ValueError(f'No rows for config={CONFIG!r}; inspect the score export and choose COMPASS_REVIEWER_CONFIG')
for c in ['risk_score', 'duration_days', 'event']:
    scores[c] = pd.to_numeric(scores[c], errors='coerce')
if scores[['risk_score', 'duration_days', 'event']].isna().any().any() or not scores.event.isin([0, 1]).all() or (scores.duration_days <= 0).any():
    raise ValueError('Invalid scores, durations, or events')
identity = ['endpoint', 'bundle', 'model_family', 'landmark_days', 'config', 'model_horizon_days', 'person_id']
if scores.duplicated(identity).any(): raise ValueError('Duplicate patient/model identities in transfer scores')
OUT.mkdir(parents=True, exist_ok=True)
display(scores.groupby(['endpoint','model_family','landmark_days','model_horizon_days'], dropna=False).agg(n=('person_id','size'), events=('event','sum')).reset_index())

## Additional patient-level inputs

The remaining analyses join the ADT longitudinal file to CRPC and AVPC annotation labels, the ADT intent classification, the Gleason timeline and sequencing data. Set environment variables below if your cluster layout differs. The optional demographics file needs one row per patient with `DFCI_MRN`, `race`, and/or `ethnicity`; categories are kept as supplied, not guessed. Missing required files stop the relevant workflow rather than silently changing the cohort. All individual-level joins stay in memory.

In [ ]:
import reviewer_sources as src
import reviewer_extended as ext
import reviewer_specificity as spec
import federated_clinical_comparison as fcc
LLM_ROOT = Path(os.getenv('LLM_ANNOTATIONS_DATA_PATH', '/data/gusev/USERS/jpconnor/data/LLM_annotations'))
LOCAL_PATH = Path(os.getenv('COMPASS_REVIEWER_LOCAL', str(DATA_ROOT / 'longitudinal_prediction_data_adt.csv')))
CRPC_PATH = Path(os.getenv('COMPASS_REVIEWER_CRPC', str(LLM_ROOT / 'LLM_disease_course/mcrpc_labels.parquet')))
AVPC_PATH = Path(os.getenv('COMPASS_REVIEWER_AVPC', str(LLM_ROOT / 'LLM_avpc/avpc_labels.parquet')))
INTENT_PATH = Path(os.getenv('COMPASS_REVIEWER_INTENT', str(DATA_ROOT / 'mrn_lists/adt_intent_labels_model_cohort.csv')))
DEATH_PATH = Path(os.getenv('COMPASS_REVIEWER_DEATH', str(DATA_ROOT / 'prostate_adt_survival_cohort_adt.csv')))
GLEASON_PATH = Path(os.getenv('COMPASS_REVIEWER_GLEASON', str(fcc.sg.DEFAULT_GLEASON_PATH)))
SOMATIC_PATH = Path(os.getenv('COMPASS_REVIEWER_SOMATIC', str(fcc.sg.DEFAULT_SOMATIC_PATH)))
SOMATIC_MANIFEST = Path(os.getenv('COMPASS_REVIEWER_SOMATIC_MANIFEST', str(fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH)))
MEDICATION_PATH = Path(os.getenv('COMPASS_REVIEWER_MEDICATIONS', '/data/gusev/PROFILE/CLINICAL/OncDRS/ALL_2025_03/MEDICATIONS.csv'))
DEMOGRAPHICS_PATH = os.getenv('COMPASS_REVIEWER_DEMOGRAPHICS')
local = src.read(LOCAL_PATH)
crpc_labels = src.read(CRPC_PATH)
avpc_labels = src.read(AVPC_PATH)
intent = src.read(INTENT_PATH)
death = src.read(DEATH_PATH)
demographics = src.read(DEMOGRAPHICS_PATH) if DEMOGRAPHICS_PATH else None
meta, gleason = src.build_patient_frame(local, crpc_labels, avpc_labels, intent, GLEASON_PATH, SOMATIC_PATH, SOMATIC_MANIFEST, death, demographics)
print(f'Patient metadata assembled: {len(meta):,}; CRPC dates: {meta.crpc_date.notna().sum():,}; demographics: {"provided" if demographics is not None else "not provided"}')

## 0. AVPC criterion overlap

Counts refer to criteria recorded among AVPC cases, not mutually exclusive causes. The C7-only count triggers the pre-specified specificity sensitivity analysis. As AVPC criterion labels and dates are derived from notes, verify the asserted lab-feature overlap in the annotation evidence before making a Methods claim.

CRPC dates come from the mCRPC labels. The AVPC labels carry their own CRPC date, so the agreement between the two sources is reported as aggregate counts.

In [ ]:
criteria = ext.criterion_counts(avpc_labels)
criteria.to_csv(OUT / 'avpc_criterion_counts.csv', index=False)
display(criteria)
if 'crpc_date' in avpc_labels:
    crpc_agreement = ext.crpc_date_agreement(crpc_labels, avpc_labels, meta.person_id)
    crpc_agreement.to_csv(OUT / 'crpc_date_agreement.csv', index=False)
    display(crpc_agreement)
else: print('AVPC labels have no crpc_date; CRPC date agreement not computed.')

## 1. Lead time

The score threshold is the landmark cohort's 80th percentile, with ties retained. The AUC bands end at 180, 365, 730 and 1,825 days; controls for a band must be event-free through 730 days or the band's end, whichever is later, so no control is censored before a case in the same band is diagnosed. Checklist flags use the frozen rule. The median lead time and fractions among eventual flagged cases are descriptive and conditional on later diagnosis. Estimates with fewer than 10 cases are suppressed. Durations reflect the transfer pipeline's administrative censoring.

In [ ]:
lead_tables, summaries = [], []
lm = PRIMARY_LANDMARK
for ep in ('avpc', 'nepc'):
    for family in ('xgboost_cox', 'elastic_net_cox', 'predictive_checklist'):
        d = scores[(scores.endpoint == ep) & (scores.model_family == family) & (scores.landmark_days == lm)].copy()
        if family == 'predictive_checklist': d = d[d.model_horizon_days.eq(365)]
        if d.empty:
            print(f'Skipped {ep} {family}: no rows at landmark {lm}')
            continue
        if d.person_id.duplicated().any(): raise ValueError(f'Multiple {family} models selected for {ep}; choose one bundle explicitly')
        d=d.merge(meta[['person_id','anchor_date','death_date']],on='person_id',how='left',validate='one_to_one')
        death_days=(d.death_date-d.anchor_date).dt.days-lm
        d['control_followup_days']=np.minimum(d.duration_days,death_days.where(death_days.ge(0),np.inf))
        summary = rq.lead_summary(d, checklist=(family == 'predictive_checklist'))
        summary.update(endpoint=ep, model_family=family, landmark_days=lm)
        if summary['events'] < MIN_EVENTS:
            for key in ('sensitivity','lead_median','lead_q25','lead_q75','fraction_180','fraction_365','ppv_730_km'): summary[key] = np.nan
        summaries.append(summary)
        if family == 'xgboost_cox':
            tab = rq.lead_auc(d, N_BOOT, SEED, MIN_EVENTS)
            tab['endpoint'] = ep
            lead_tables.append(tab)
if lead_tables:
    lead_auc = pd.concat(lead_tables, ignore_index=True)
    lead_auc.to_csv(OUT / 'lead_time_auc.csv', index=False)
    rq.draw_lead(lead_auc, OUT / 'lead_time_auc.pdf')
    display(lead_auc)
if summaries:
    lead_summary = pd.DataFrame(summaries)
    lead_summary.to_csv(OUT / 'lead_time_summary.csv', index=False)
    display(lead_summary)

## 3. Calibration on platinum

The transfer implementation defines elastic-net `risk_score` as the design matrix multiplied by the fitted coefficients and XGBoost `risk_score` as the output margin. Thus the elastic-net score is the linear predictor expected by `predict_survival`; the XGBoost margin has no exported absolute baseline. Three elastic-net versions are reported at 1, 2 and 3 years: the transferred model, a DFCI baseline update with the coefficients fixed as an offset, and a DFCI update of both slope and baseline. The updates are fitted and evaluated in DFCI and are **not external validation**. Calibration groups are deciles of predicted 2-year risk, pooled with neighbouring deciles until each has at least 10 events; a KM-only Brier reference uses the same observed risk. Intervals use patient bootstrap resampling. The XGBoost Cox slope below is fitted on the **unstandardized** margin.

In [ ]:
run_path = TRANSFER_DIR / 'transfer_run.json'
if not run_path.exists(): raise FileNotFoundError(f'Bundle provenance missing: {run_path}')
run = json.loads(run_path.read_text())
cal = scores[(scores.endpoint == 'platinum') & (scores.model_family == 'elastic_net_cox') & (scores.landmark_days == PRIMARY_LANDMARK)].copy()
candidates = []
if cal.empty: print(f'No platinum elastic-net rows at day {PRIMARY_LANDMARK}; calibration skipped.')
else:
    if cal.person_id.duplicated().any() or cal.bundle.nunique() != 1: raise ValueError('Select exactly one elastic-net bundle')
    stem = cal.bundle.iloc[0]
    paths = run['bundles'][stem]['paths']
    bundle = merge_bundles([load_bundle(p) for p in paths], sources=paths)
    candidates = [m for m in bundle['models'] if int(m['landmark_days']) == PRIMARY_LANDMARK and m.get('config') == CONFIG]
    if len(candidates) != 1: raise ValueError(f'Expected one elastic-net model; found {len(candidates)}')
    cal_summary, cal_points = ext.calibration_versions(cal, candidates[0], n_boot=N_BOOT, seed=SEED, time_unit_days=bundle.get('time_unit_days') or 1.0)
    if cal_summary.empty: print('Calibration not estimable: fewer than 10 events.')
    else:
        cal_summary.to_csv(OUT / 'calibration_all_versions.csv', index=False)
        cal_points.to_csv(OUT / 'calibration_all_deciles.csv', index=False)
        ext.plot_calibration(cal_points, OUT / 'calibration_all_versions.pdf', 'Platinum calibration at 2 years')
        display(cal_summary); display(cal_points)
slopes = []
for ep in ('platinum','avpc','nepc'):
    d = scores[(scores.endpoint == ep) & (scores.model_family == 'xgboost_cox') & (scores.landmark_days == PRIMARY_LANDMARK)]
    if not d.empty:
        if d.person_id.duplicated().any(): raise ValueError('Multiple XGBoost models selected')
        lo,hi = rq.ci_bootstrap(d,rq.cox_unstandardized,N_BOOT,np.random.default_rng(SEED)) if d.event.sum() >= MIN_EVENTS else (np.nan,np.nan)
        slopes.append(dict(endpoint=ep, n=len(d), events=int(d.event.sum()), unstandardized_cox_slope=rq.cox_unstandardized(d),lower=lo,upper=hi))
if slopes:
    slopes = pd.DataFrame(slopes); slopes.to_csv(OUT / 'xgboost_cox_slopes.csv', index=False); display(slopes)

## 1c–d. Milestones, flags across landmarks and date-precision sensitivity

Gaps are measured as milestone date minus the landmark date: positive values mean a milestone occurred after the flag. Cumulative flagging uses the union of patients with an observed event in the available landmark frames and should be read with the number of eligible cases at each landmark. Month/year precision is excluded only when explicitly identified; unknown precision is reported separately. AVPC labels do not export an AVPC diagnosis precision field in the current schema, so the AVPC analysis excludes cases by **CRPC** date precision. That is only a proxy for the precision of the AVPC date and should be described as such.

In [ ]:
guideline_path = TRANSFER_DIR / 'all_models' / 'transfer_guideline_scores.csv'
guidelines = pd.read_csv(guideline_path, dtype={'person_id':'string'}) if guideline_path.exists() else pd.DataFrame()
if not guidelines.empty:
    guidelines = guidelines[guidelines.config.eq(CONFIG)].copy()
    guidelines['person_id'] = pd.to_numeric(guidelines.person_id, errors='raise').astype('Int64').astype('string')
    for col in ('risk_score','duration_days','event','flag','model_horizon_days','landmark_days'): guidelines[col] = pd.to_numeric(guidelines[col],errors='coerce')
    # Generalized rules have shifted item counts: the original calibrator is not valid.
    guidelines['model_family'] = 'guideline_checklist'
else: print('Generalized checklist scores missing; rerun the guideline cell in updated notebook 04.')
flag_scores = pd.concat([scores, guidelines], ignore_index=True)
milestone_tables=[]; milestone_summaries=[]; lead_plot=[]; cumulative=[]; sensitivities=[]; death_audits=[]; generalized_summaries=[]
for ep in ('avpc','nepc'):
    for family in ('xgboost_cox','elastic_net_cox','predictive_checklist','guideline_checklist'):
        lm=PRIMARY_LANDMARK
        d=flag_scores[(flag_scores.endpoint==ep)&(flag_scores.model_family==family)&(flag_scores.landmark_days==lm)].copy()
        if 'checklist' in family: d=d[d.model_horizon_days.eq(365)]
        if d.empty: continue
        if d.person_id.duplicated().any(): raise ValueError(f'Multiple {ep} {family} score models')
        d['flagged'] = d.flag.eq(1) if 'checklist' in family else d.risk_score.ge(d.risk_score.quantile(.8))
        if family=='guideline_checklist':
            gs=rq.lead_summary(d,checklist=True); gs.update(endpoint=ep,model_family=family,landmark_days=lm); generalized_summaries.append(gs)
        d=d.merge(meta,on='person_id',how='left',validate='one_to_one')
        d['landmark_date']=d.anchor_date+pd.to_timedelta(lm,unit='D')
        audit=ext.lead_death_audit(d); audit.update(endpoint=ep,model_family=family); death_audits.append(audit)
        m, sm, flagged = ext.milestones(d,ep)
        m['model_family']=family; sm['model_family']=family
        milestone_tables.append(m); milestone_summaries.append(sm)
        if family=='xgboost_cox': lead_plot.append((ep, flagged))
        # Precision fields are captured only where actually exported.
        precision_cols=['crpc_date_precision']
        if ep=='nepc' and 'NEPC_DATE_PRECISION' in local:
            prec=src.one_patient(local,['NEPC_DATE_PRECISION'],'NEPC precision')
            d=d.merge(prec,on='person_id',how='left',validate='one_to_one'); precision_cols.append('NEPC_DATE_PRECISION')
        known_coarse=pd.Series(False,index=d.index)
        for col in precision_cols: known_coarse |= d[col].astype('string').str.lower().isin(['month','year'])
        death_days=(d.death_date-d.landmark_date).dt.days
        d['control_followup_days']=np.minimum(d.duration_days,death_days.where(death_days.ge(0),np.inf))
        clean=d[~(d.event.eq(1)&known_coarse)]
        result=rq.lead_summary(clean,checklist=('checklist' in family)); result.update(endpoint=ep,model_family=family,excluded_cases=int((d.event.eq(1)&known_coarse).sum()), unknown_precision_cases=int((d.event.eq(1)&d[precision_cols].isna().any(axis=1)).sum()))
        sensitivities.append(result)
        if family=='xgboost_cox':
            a=rq.lead_auc(clean,N_BOOT,SEED); a['endpoint']=ep; a['sensitivity_population']='exclude known month/year'
            a.to_csv(OUT / f'lead_precision_{ep}.csv',index=False)
    for family in ('xgboost_cox','elastic_net_cox','predictive_checklist','guideline_checklist'):
        cumulative.append(ext.cumulative_flags(flag_scores,ep,family))
if generalized_summaries:
    pd.DataFrame(generalized_summaries).to_csv(OUT / 'lead_time_generalized_summary.csv',index=False)
    display(pd.DataFrame(generalized_summaries))
if death_audits: pd.DataFrame(death_audits).to_csv(OUT / 'lead_control_death_audit.csv',index=False); display(pd.DataFrame(death_audits))
if milestone_tables: pd.concat(milestone_tables).to_csv(OUT / 'milestones.csv',index=False)
if milestone_summaries: display(pd.DataFrame(milestone_summaries)); pd.DataFrame(milestone_summaries).to_csv(OUT / 'milestone_summary.csv',index=False)
if cumulative: display(pd.concat(cumulative)); pd.concat(cumulative).to_csv(OUT / 'cumulative_flags.csv',index=False)
if sensitivities: pd.DataFrame(sensitivities).to_csv(OUT / 'lead_precision_summary.csv',index=False)
if lead_plot:
    fig,axes=plt.subplots(1,len(lead_plot),figsize=(6*len(lead_plot),4),squeeze=False)
    for ax,(ep,d) in zip(axes[0],lead_plot):
        gaps=d[ep+'_date_gap'].dropna(); crpc=d.crpc_date_gap.dropna()
        if len(gaps)>=10: ax.hist(gaps/30.4375,bins=25,alpha=.65,label='Diagnosis')
        if len(crpc)>=10: ax.axvline(crpc.median()/30.4375,color='darkred',label='Median CRPC')
        ax.axvline(0,color='grey',ls='--'); ax.set(title=ep.upper(),xlabel='Months after landmark',ylabel='Flagged cases')
        if ax.get_legend_handles_labels()[0]: ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(OUT / 'lead_milestones.pdf'); plt.close(fig)

## 2a. AVPC or NEPC within CRPC

The primary outcome is the composite of AVPC or NEPC, whichever is first, by CRPC + 730 days. AVPC and NEPC are secondary endpoint-specific comparisons, in which patients with only the other phenotype are excluded. The cohort includes patients whose CRPC is documented strictly after the day-180 landmark; the composite uses patients in both the AVPC and NEPC incident landmark frames, whose XGBoost scores must agree. A control has neither phenotype by CRPC + 730 days and at least 730 days of follow-up. Clinical variables use PSA and Gleason available by CRPC; metastatic status at CRPC is 'yes' for a metastasis dated by CRPC, 'no' for a later metastasis or a non-metastatic label, and 'date unknown' for a metastatic label without a date. The C7-only sensitivity analysis removes cases that met AVPC by criterion C7 alone, except composite cases that are also NEPC cases. The prediction comparison is stratified five-fold and penalized; the adjusted odds ratio and likelihood-ratio test are separate unpenalized, in-sample fits. Bootstrap intervals compare the fixed out-of-fold predictions, so they do not include model-refitting uncertainty.

In [ ]:
xgb_frames={}
for ep in ('avpc','nepc'):
    d=scores[(scores.endpoint==ep)&scores.model_family.eq('xgboost_cox')&scores.landmark_days.eq(PRIMARY_LANDMARK)].copy()
    if d.person_id.duplicated().any(): raise ValueError('Multiple XGBoost models selected')
    xgb_frames[ep]=d[['person_id','risk_score']]
# One model scores both frames, so a patient eligible for both must have one score.
both=xgb_frames['avpc'].merge(xgb_frames['nepc'],on='person_id',suffixes=('','_nepc'),validate='one_to_one')
if not np.allclose(both.risk_score,both.risk_score_nepc): raise ValueError('AVPC and NEPC XGBoost scores differ for the same patient')
xgb_frames['avpc_or_nepc']=both[['person_id','risk_score']]
criteria_cols=[f'c{i}_met' for i in range(1,8)]
crpc_results=[]; audits=[]
for ep in ('avpc_or_nepc','avpc','nepc'):  # composite is primary
    d=xgb_frames[ep].merge(meta,on='person_id',how='inner',validate='one_to_one')
    d['landmark_date']=d.anchor_date+pd.to_timedelta(PRIMARY_LANDMARK,unit='D')
    for timing in (True,False):
        for horizon in (730,365):
            # The 365-day run is a pre-specified fallback, never silently substituted.
            sub=spec.define_crpc_outcome(d,ep,horizon,after_landmark=timing)
            for label,n in sub.exclusion.value_counts().items(): audits.append(dict(endpoint=ep,timing='after' if timing else 'before_or_at',horizon=horizon,category=label,n=int(n)))
            result=spec.fit_specificity(sub,N_BOOT,SEED,MIN_EVENTS)
            result.update(endpoint=ep,primary=(ep=='avpc_or_nepc' and timing and horizon==730),crpc_timing='after' if timing else 'before_or_at',horizon=horizon,sensitivity='all cases')
            crpc_results.append(result)
            if timing and horizon==730 and ep!='nepc':
                flags=sub[criteria_cols].fillna(False).astype(bool)
                only7=flags.c7_met & flags.sum(axis=1).eq(1) & sub.outcome.eq(1)
                if ep=='avpc_or_nepc': only7&=~spec.define_crpc_outcome(d,'nepc',horizon,after_landmark=timing).outcome.eq(1)
                if only7.any():
                    sens=spec.fit_specificity(sub.loc[~only7],N_BOOT,SEED,MIN_EVENTS)
                    sens.update(endpoint=ep,primary=False,crpc_timing='after',horizon=horizon,sensitivity='exclude C7-only AVPC cases',excluded_cases=int(only7.sum()))
                    crpc_results.append(sens)
crpc_results=pd.DataFrame(crpc_results); audits=pd.DataFrame(audits)
crpc_results.to_csv(OUT / 'crpc_specificity.csv',index=False); audits.to_csv(OUT / 'crpc_cohort_audit.csv',index=False)
display(crpc_results); display(audits)

## 3a–c. Checklist calibration

Both checklists are evaluated at the day-180 landmark used in Figure 6. The direct checklist is compared with its exported probability at its own horizon; score totals are pooled with neighbouring totals until each group has at least 10 events. The generalized (rounded-threshold) checklist changes item counts, so the exact-score calibrator does not apply: it is reported with observed risk by score total and flag only, with no predicted risk or O/E. The elastic-net calibration, including the DFCI-updated versions, is in section 3; the XGBoost slopes there have no absolute risk.

In [ ]:
checklist_tables=[]
checks=scores[(scores.endpoint=='platinum')&scores.model_family.eq('predictive_checklist')&scores.landmark_days.eq(PRIMARY_LANDMARK)]
if not checks.empty:
    direct=ext.checklist_calibration(checks,N_BOOT,SEED).assign(checklist='direct')
    ext.plot_calibration(direct[direct.grouping.eq('risk_score')],OUT / 'checklist_calibration.pdf','Checklist calibration')
    checklist_tables.append(direct)
else: print(f'No platinum checklist rows at day {PRIMARY_LANDMARK}')
general=guidelines[guidelines.endpoint.eq('platinum')&guidelines.landmark_days.eq(PRIMARY_LANDMARK)] if not guidelines.empty else guidelines
if not general.empty:
    checklist_tables.append(ext.checklist_calibration(general,N_BOOT,SEED,observed_only=True).assign(checklist='generalized (observed only)'))
if checklist_tables:
    checklist_cal=pd.concat(checklist_tables,ignore_index=True)
    checklist_cal.to_csv(OUT / 'checklist_calibration.csv',index=False)
    display(checklist_cal)

## 3d. Subgroup discrimination and heterogeneity

Every endpoint uses its day-180 XGBoost cohort; platinum O/E uses elastic-net predictions on the matching subgroup. Categories unavailable in source data are not manufactured. Subgroup hazard ratios are per SD of the endpoint's full-cohort score, the same scale as the interaction test, so they can be compared across levels. The interaction test compares nested Cox models, not confidence intervals. Metastatic status at the ADT anchor is 'date unknown' when the label says metastatic but no date is recorded, rather than 'no'. Sequenced means present in the supplied sequencing data, irrespective of the sequencing date.

In [ ]:
gleason_lm=src.gleason_at_landmark(meta,gleason,PRIMARY_LANDMARK)
submeta=meta.merge(gleason_lm,on='person_id',how='left',validate='one_to_one')
submeta['age_group']=pd.cut(pd.to_numeric(submeta.age,errors='coerce'),[-np.inf,65,75,np.inf],right=False,labels=['<65','65–74','≥75']).astype('string').fillna('missing')
submeta['anchor_year']=np.where(submeta.anchor_date.isna(),'missing',np.where(submeta.anchor_date.dt.year<2018,'<2018','≥2018'))
submeta['metastatic_at_anchor']=spec.metastatic_status(submeta.met_date,submeta.is_metastatic,submeta.anchor_date).to_numpy()
g=pd.to_numeric(submeta.gleason_at_landmark,errors='coerce'); submeta['gleason_group']=np.where(g.isna(),'missing',np.where(g>=8,'≥8','≤7'))
variables=['age_group','adt_intent','metastatic_at_anchor','anchor_year','gleason_group','sequenced','race','ethnicity']
subtables=[]; intertables=[]
for ep in ('platinum','avpc','nepc'):
    d=scores[(scores.endpoint==ep)&scores.model_family.eq('xgboost_cox')&scores.landmark_days.eq(PRIMARY_LANDMARK)].merge(submeta,on='person_id',how='left',validate='one_to_one')
    if d.empty: continue
    table,inter=ext.subgroup_metrics(d,variables,n_boot=N_BOOT,seed=SEED); table['endpoint']=ep; inter['endpoint']=ep
    subtables.append(table); intertables.append(inter)
if subtables:
    subgroups=pd.concat(subtables,ignore_index=True); interaction=pd.concat(intertables,ignore_index=True)
    subgroups.to_csv(OUT / 'subgroup_metrics.csv',index=False); interaction.to_csv(OUT / 'subgroup_interactions.csv',index=False)
    forest=subgroups.copy(); forest['label']=forest.endpoint.str.upper()+' · '+forest.variable+' · '+forest.level
    ext.plot_forest(forest,OUT / 'subgroup_forest.pdf','Subgroup association')
    display(subgroups); display(interaction)
# Elastic-net calibration within platinum subgroups uses its own cohort and bundle.
if not cal.empty and len(candidates)==1:
    ed=cal.merge(submeta,on='person_id',how='left',validate='one_to_one')
    ed['predicted_730']=1-predict_survival(ed.risk_score.to_numpy(),candidates[0],[730],time_unit_days=bundle.get('time_unit_days') or 1.0)['survival_730d'].to_numpy()
    et,_=ext.subgroup_metrics(ed,variables,n_boot=N_BOOT,seed=SEED,probabilities='predicted_730')
    et.to_csv(OUT / 'subgroup_elastic_net.csv',index=False)

## 2c. Local comparison endpoints

The same frozen day-180 XGBoost model is scored on each endpoint's own incident landmark cohort using the transfer feature builder. Therapy events use exact preferred medication names; death dates come from the survival cohort. The docetaxel comparator is the first docetaxel start on or after the CRPC date, i.e. treatment for castration resistance; any docetaxel start (which includes hormone-sensitive use) and the first ARPI start are sensitivity comparators. All hazard ratios are per SD of one reference distribution, the patient-level scores across the comparison frames, so they are on a common scale. A second table restricts the primary endpoints to the patients eligible for all of them, to compare associations on the same people; it reports each endpoint's own-cohort N alongside. A third table treats the first new systemic therapy class after the landmark as a cause-specific outcome, censoring at a different first new class. A class is new only if its first-ever start is after the landmark, so continued treatment never counts. Same-day starts in different classes are excluded from this secondary analysis rather than arbitrarily ordered. Associations are descriptive and do not establish disease specificity.

In [ ]:
if not run_path.exists(): raise FileNotFoundError(run_path)
xgb_stems=[stem for stem,info in run['bundles'].items() if info.get('model_family')=='xgboost_cox']
if len(xgb_stems)!=1: raise ValueError(f'Expected one XGBoost bundle in provenance, found {xgb_stems}')
xgb_stem=xgb_stems[0]; xp=run['bundles'][xgb_stem]['paths']
xgb_bundle=merge_bundles([load_bundle(p) for p in xp],sources=xp)
xgb_primary=dict(xgb_bundle); xgb_primary['models']=[m for m in xgb_bundle['models'] if m.get('config')==CONFIG and int(m['landmark_days'])==PRIMARY_LANDMARK]
if len(xgb_primary['models'])!=1: raise ValueError(f'Expected one XGBoost model at landmark {PRIMARY_LANDMARK}')
meds=src.medication_dates(MEDICATION_PATH,meta.person_id)
lu=meta[['person_id','lu_psma_date']].dropna().rename(columns={'lu_psma_date':'med_date'}).assign(therapy='lu_psma')
meds=pd.concat([meds,lu],ignore_index=True)
crpc_dates=meta.set_index('person_id').crpc_date.dropna()
event_dates={
    'docetaxel': ext.first_start(meds,'docetaxel',after=crpc_dates),
    'cabazitaxel': ext.first_start(meds,'cabazitaxel'),
    'radium': ext.first_start(meds,'radium'),
    'lu_psma': ext.first_start(meds,'lu_psma'),
    'death': meta[['person_id','death_date']].rename(columns={'death_date':'event_date'}).dropna(),
    # Sensitivity comparators.
    'docetaxel_any_start': ext.first_start(meds,'docetaxel'),
    'arpi': ext.first_start(meds,'arpi'),
}
new_scores=[]
for ep,dates in event_dates.items():
    if dates.empty:
        print(f'{ep}: no dated events; not estimable'); continue
    z=ext.score_comparison_endpoint(local,meta,ep,dates,xgb_stem,xgb_primary,landmark=PRIMARY_LANDMARK)
    if not z.empty: new_scores.append(z)
existing=scores[(scores.endpoint.isin(['avpc','nepc','platinum']))&scores.model_family.eq('xgboost_cox')&scores.landmark_days.eq(PRIMARY_LANDMARK)]
comparison=pd.concat([existing,*new_scores],ignore_index=True)
# One model and one feature build: a patient's score should not depend on the endpoint frame.
spread=comparison.groupby('person_id').risk_score.agg(lambda s:s.max()-s.min())
print(f'Largest within-patient score difference across endpoint frames: {spread.max():.3g}')
reference=comparison.drop_duplicates('person_id').risk_score
SCORE_CENTER,SCORE_SD=reference.mean(),reference.std(ddof=0)
comparison_table=ext.comparison_metrics(comparison,N_BOOT,SEED,SCORE_CENTER,SCORE_SD)
comparison_table.to_csv(OUT / 'comparison_endpoints.csv',index=False)
ext.plot_forest(comparison_table,OUT / 'comparison_endpoints.pdf','Local comparison endpoints')
COMMON_ENDPOINTS=['platinum','avpc','nepc','docetaxel','cabazitaxel','radium','lu_psma','death']
present=[ep for ep in COMMON_ENDPOINTS if comparison.endpoint.eq(ep).any()]
common=set.intersection(*(set(comparison.loc[comparison.endpoint.eq(ep),'person_id']) for ep in present))
matched=ext.comparison_metrics(comparison[comparison.endpoint.isin(present)&comparison.person_id.isin(common)],N_BOOT,SEED,SCORE_CENTER,SCORE_SD)
own=comparison_table.set_index('endpoint')[['n','events']].rename(columns={'n':'n_own_cohort','events':'events_own_cohort'})
matched=matched.merge(own,left_on='endpoint',right_index=True,how='left')
matched.to_csv(OUT / 'comparison_endpoints_common_cohort.csv',index=False)
if lead_tables: ext.main_figure(lead_auc,matched,OUT / 'main_lead_and_comparison.pdf')
print(f'Common eligible cohort: {len(common):,} of {comparison.person_id.nunique():,} scored patients; endpoints: {", ".join(present)}')
display(comparison_table); display(matched)

In [ ]:
# First new therapy class after the landmark: a class counts only if its first-ever start is after it.
therapy,tied=ext.first_new_therapy(meds,meta,PRIMARY_LANDMARK)
print('Same-day first-class ties excluded:',len(tied))
local_no_ties=local[~pd.to_numeric(local.DFCI_MRN,errors='coerce').astype('Int64').astype('string').isin(tied)]
cause_scores=[]
for cls in ('platinum','taxane','radium','lu_psma','arpi'):
    target=therapy[therapy.therapy_class.eq(cls)][['person_id','med_date']].rename(columns={'med_date':'event_date'})
    other=therapy[~therapy.therapy_class.eq(cls)][['person_id','med_date']].rename(columns={'med_date':'competing_date'})
    if target.empty: print(f'{cls}: no events'); continue
    z=ext.score_comparison_endpoint(local_no_ties,meta,cls,target,xgb_stem,xgb_primary,landmark=PRIMARY_LANDMARK,competing=other)
    if not z.empty: cause_scores.append(z)
if cause_scores:
    cause_table=ext.comparison_metrics(pd.concat(cause_scores,ignore_index=True),N_BOOT,SEED,SCORE_CENTER,SCORE_SD)
    cause_table.to_csv(OUT / 'first_therapy_cause_specific.csv',index=False)
    ext.plot_forest(cause_table,OUT / 'first_therapy_cause_specific.pdf','First new therapy class: cause-specific')
    display(cause_table)

## 2b. Exploratory scoring at CRPC

The ADT-trained bundles are shifted to a different decision point, so this is an exploratory distribution shift, not the primary validation. CRPC is the new anchor, lab features are built before that anchor, and prevalent AVPC/NEPC at or before CRPC are excluded by the transfer's standard incident-event gate. That gate also drops events dated on the CRPC day itself (duration 0); the timing table counts events before, on and after the CRPC date so the size of that loss is reported.

In [ ]:
crpc_timing=ext.crpc_anchor_audit(meta)
crpc_timing.to_csv(OUT / 'crpc_reanchored_event_timing.csv',index=False); display(crpc_timing)
crpc_local=ext.reanchor_local(local,meta)
if crpc_local.empty: print('No dated CRPC anchors: exploratory rescoring not estimable')
else:
    families={}
    for stem,info in run['bundles'].items():
        if info.get('model_family') not in ('xgboost_cox','elastic_net_cox'): continue
        paths=info['paths']; b=merge_bundles([load_bundle(p) for p in paths],sources=paths)
        b=dict(b); b['models']=[m for m in b['models'] if int(m['landmark_days'])==0 and m.get('config')==CONFIG]
        if b['models']: families[stem]=b
    if not families: raise ValueError('No landmark-0 bundles for exploratory CRPC scoring')
    res_crpc=fet.evaluate_transfer(crpc_local,families,endpoints=('avpc','nepc','platinum'),landmarks=(0,),
        lab_mapping=fet.load_lab_mapping(fet.DEFAULT_LAB_MAPPING),n_bootstrap=N_BOOT,seed=SEED)
    metrics=res_crpc['metrics']; metrics=metrics[metrics.n_events.ge(MIN_EVENTS)].copy()
    metrics['analysis']='exploratory: CRPC reanchored'; metrics.to_csv(OUT / 'crpc_reanchored_transfer.csv',index=False)
    display(metrics)

## Interpretation and methods notes

This analysis uses patients eligible under the transferred model's landmark frame, not a prospective deployment cohort. Event documentation may lag biological onset. KM and cause-specific estimates treat competing death as censoring, so the resulting quantities are not cumulative incidence in the presence of competing events. OOF model-comparison intervals condition on the fitted out-of-fold predictions. Report counts, missing inputs, the event floor, and whether estimates came from transported or locally updated models. Date precision sensitivity cannot remove unrecorded imprecision, and AVPC's current exported label has no AVPC-date precision. Race and ethnicity require a verified source and must not be reported as if missing values were a demographic category. Lead-time controls in the 731–1,825-day band need 1,825 days of event-free follow-up, so that band rests on the longest-followed patients. The generalized checklist has no valid probability calibrator and is reported with observed risk only.